# MenuScan TN · 03.03 Évaluation des modèles et dossier J8 (T7.4)

Autrice : Sarra Abdeljaouad

**Objet.**
1. Rassembler en un seul endroit les mesures des trois modèles de la phase 7 : registre des modèles.
2. Fixer la règle des **candidats à la formule Commande** (tuile de TB-4).
3. Préparer les chiffres du **jalon J8** pour la décision du sponsor.

Ce notebook ne réentraîne rien : il lit les sorties de `02_prediction_resiliation`, `01_regles_association` et `02_sensibilite_prix`.

In [1]:
import sys
sys.path.insert(0, "../00_commun")

import json

import numpy as np
import pandas as pd

from menuscan_analytics import config

config.style()
T = config.DOSSIER_TABLES
lire_csv = lambda nom: pd.read_csv(T / f"{nom}.csv", sep=";", encoding="utf-8-sig")
fiche = json.loads((config.DOSSIER_MODELES / "fiches" / "resiliation.json").read_text(encoding="utf-8"))

## 1. Candidats à la formule Commande

**Règle, fixée ici une fois pour toutes.** Un établissement est candidat au passage de la formule Essentiel à la formule Commande s'il remplit trois conditions au 31/12/2026 :
1. il est actif et en formule Essentiel ;
2. son usage du menu sur 3 mois (sessions par table et par jour) est **au moins égal à la médiane des établissements déjà en formule Commande** : ses clients consultent le menu autant que ceux qui commandent déjà à table ;
3. il **n'est pas en alerte de résiliation** (K19) : on ne propose pas une formule plus chère à un client qu'on risque de perdre.

In [2]:
panel = pd.read_parquet(config.DOSSIER_FEATURES / "etablissement_mois.parquet")
dec = panel[panel.sous_jeu == "a_scorer"]
scores = lire_csv("ML02_scores_2026_12")
seuil = dec.loc[dec.formule == "Commande", "sessions_table_jour_3m"].median()
cand = (dec[(dec.formule == "Essentiel") & (dec.sessions_table_jour_3m >= seuil)]
        [["etablissement_id", "sessions_table_jour_3m"]]
        .merge(scores[["etablissement_id", "probabilite", "alerte"]], on="etablissement_id", how="left"))
exclus_alerte = int(cand.alerte.sum())
cand = cand[~cand.alerte].drop(columns="alerte").rename(columns={"probabilite": "probabilite_resiliation"})
cand["seuil_usage"] = seuil
cand = cand.sort_values("sessions_table_jour_3m", ascending=False).reset_index(drop=True)
print(f"Seuil d'usage (médiane Commande) : {seuil:.3f} session par table et par jour")
print(f"Établissements Essentiel actifs : {(dec.formule == 'Essentiel').sum()} ; au-dessus du seuil : "
      f"{len(cand) + exclus_alerte} ; exclus car en alerte : {exclus_alerte} ; candidats : {len(cand)}")
config.enregistrer_table(cand, "ML03_candidats_commande", index=False)
cand.head(10)

Seuil d'usage (médiane Commande) : 0.116 session par table et par jour
Établissements Essentiel actifs : 107 ; au-dessus du seuil : 48 ; exclus car en alerte : 1 ; candidats : 47


,etablissement_id,sessions_table_jour_3m,probabilite_resiliation,seuil_usage
0,ETB0177,0.624,0.001,0.116
1,ETB0323,0.490,0.006,0.116
2,ETB0263,0.441,0.008,0.116
3,ETB0271,0.385,0.000,0.116
4,ETB0011,0.380,0.004,0.116
5,ETB0339,0.341,0.070,0.116
6,ETB0006,0.326,0.040,0.116
7,ETB0128,0.316,0.026,0.116
8,ETB0313,0.297,0.013,0.116
9,ETB0298,0.291,0.012,0.116


## 2. Registre des modèles

In [3]:
regles_cat, regles_plat = lire_csv("DM01_regles_categories"), lire_csv("DM01_regles_plats")
el = lire_csv("DM02_elasticites_categories")
mesures_test = lire_csv("ML02_mesures_test_2026").set_index(lire_csv("ML02_mesures_test_2026").columns[0])["valeur"]
VERSION = "v1 (27/11/2026)"
L = []
def ajouter(modele, algo, mesure, valeur, bas=None, haut=None, decision=None):
    L.append({"modele": modele, "version": VERSION, "algorithme": algo, "mesure": mesure, "valeur": valeur,
              "bas_95": bas, "haut_95": haut, "decision": decision})

algo_r = fiche["algorithme"]
ajouter("resiliation", algo_r, "AUC validation 2025", fiche["auc_validation"])
ajouter("resiliation", algo_r, "AUC test 2026", fiche["auc_test_2026"], *fiche["auc_test_ic95"],
        decision=f"J8 {fiche['jalon_J8'].lower()} en valeur centrale ; borne basse sous 0,75")
ajouter("resiliation", algo_r, "AUC règle simple 2026", fiche["auc_regle_simple_test"])
ajouter("resiliation", algo_r, "Précision de l'alerte 2026", fiche["precision_alerte_test"])
ajouter("resiliation", algo_r, "Rappel de l'alerte 2026", fiche["rappel_alerte_test"])
algo_a = "lift ajusté par établissement, test de Poisson (Bonferroni)"
ajouter("associations", algo_a, "Paires de catégories retenues", len(regles_cat))
ajouter("associations", algo_a, "Paires de plats retenues", len(regles_plat))
ajouter("associations", algo_a, "Part des paires confirmées en 2026",
        (pd.concat([regles_cat, regles_plat]).lift_2026 > 1).mean())
glob = lire_csv("DM02_elasticite_globale").iloc[0]
algo_e = "Poisson conditionnelle, effets fixes article et cellule"
ajouter("elasticite", algo_e, "Élasticité toutes catégories", glob.elasticite, glob.bas_95, glob.haut_95)
ajouter("elasticite", algo_e, "Catégories exploitables", int((el.lecture != "non concluant").sum()))
ajouter("elasticite", algo_e, "Catégories non concluantes", int((el.lecture == "non concluant").sum()))
registre = pd.DataFrame(L)
config.enregistrer_table(registre, "ML03_registre_modeles", index=False)
registre

,modele,version,algorithme,mesure,valeur,bas_95,haut_95,decision
0,resiliation,v1 (27/11/2026),"logistique, C = 0.03, classes pondérées",AUC validation 2025,0.862,NaN,NaN,NaN
1,resiliation,v1 (27/11/2026),"logistique, C = 0.03, classes pondérées",AUC test 2026,0.794,0.738,0.850,J8 atteint en valeur centrale ; borne basse so...
2,resiliation,v1 (27/11/2026),"logistique, C = 0.03, classes pondérées",AUC règle simple 2026,0.668,NaN,NaN,NaN
3,resiliation,v1 (27/11/2026),"logistique, C = 0.03, classes pondérées",Précision de l'alerte 2026,0.147,NaN,NaN,NaN
4,resiliation,v1 (27/11/2026),"logistique, C = 0.03, classes pondérées",Rappel de l'alerte 2026,0.487,NaN,NaN,NaN
5,associations,v1 (27/11/2026),"lift ajusté par établissement, test de Poisson...",Paires de catégories retenues,3.000,NaN,NaN,NaN
6,associations,v1 (27/11/2026),"lift ajusté par établissement, test de Poisson...",Paires de plats retenues,10.000,NaN,NaN,NaN
7,associations,v1 (27/11/2026),"lift ajusté par établissement, test de Poisson...",Part des paires confirmées en 2026,1.000,NaN,NaN,NaN
8,elasticite,v1 (27/11/2026),"Poisson conditionnelle, effets fixes article e...",Élasticité toutes catégories,-0.698,-0.888,-0.478,NaN
9,elasticite,v1 (27/11/2026),"Poisson conditionnelle, effets fixes article e...",Catégories exploitables,6.000,NaN,NaN,NaN


## 3. Dossier du jalon J8

Critère du plan de projet : **AUC d'au moins 0,75 sur 2026**, mesurée sur des données que le modèle n'a pas vues.

In [4]:
j8 = pd.Series({
    "AUC 2026 (valeur centrale)": fiche["auc_test_2026"],
    "Intervalle à 95 %": f"{fiche['auc_test_ic95'][0]:.3f} à {fiche['auc_test_ic95'][1]:.3f}",
    "Seuil du jalon": 0.75,
    "Seuil atteint en valeur centrale": fiche["auc_test_2026"] >= 0.75,
    "Seuil atteint par la borne basse": fiche["auc_test_ic95"][0] >= 0.75,
    "Gain sur la meilleure règle simple": round(fiche["auc_test_2026"] - fiche["auc_regle_simple_test"], 3),
    "Résiliations dans le test 2026": 76,
})
config.enregistrer_table(j8.to_frame("valeur"), "ML03_dossier_j8")
j8.to_frame("valeur")

,valeur
AUC 2026 (valeur centrale),0.794
Intervalle à 95 %,0.738 à 0.850
Seuil du jalon,0.750
Seuil atteint en valeur centrale,True
Seuil atteint par la borne basse,False
Gain sur la meilleure règle simple,0.126
Résiliations dans le test 2026,76


**Options pour le sponsor.**

| Option | Contenu | Conséquence |
|---|---|---|
| A. Valider J8 | Le seuil est atteint en valeur centrale, et le modèle bat nettement la règle simple | Les scores K19 sont publiés dans TB-4 ; suivi mensuel de l'AUC sur 2027 |
| B. Valider avec réserve | Comme A, avec une revue après 6 mois de 2027 : si l'AUC glissante passe sous 0,75, le modèle est réappris | Même publication ; la réserve est tracée au registre des décisions |
| C. Refuser | La borne basse est sous 0,75 : la preuve est jugée insuffisante | TB-4 n'affiche pas K19 ; nouvelle tentative avec plus d'historique |

**Recommandation : B.**
- La valeur centrale dépasse le seuil.
- Le modèle repère près de la moitié des résiliations en n'alertant que sur 10 % des établissements.
- Mais l'intervalle est large : seulement 76 résiliations en test. La réserve encadre ce risque sans bloquer l'usage.

La décision revient au sponsor.